In [ ]:
import json, urllib.request, zipfile, io, subprocess, sys
from pathlib import Path
from packaging.tags import sys_tags
from packaging.utils import parse_wheel_filename
_demo_root = Path('.hyper-demo').resolve()
_demo_tools = _demo_root / 'tools'
_demo_packages = _demo_root / 'packages'
_tags = set(sys_tags())
with urllib.request.urlopen('https://pypi.org/pypi/uv/json') as response:
    _uv_release = json.load(response)
_uv_wheel = next(f for f in _uv_release['urls'] if f['filename'].endswith('.whl') and parse_wheel_filename(f['filename'])[3] & _tags)
with urllib.request.urlopen(_uv_wheel['url']) as response:
    _wheel_bytes = response.read()
import hashlib
assert hashlib.sha256(_wheel_bytes).hexdigest() == _uv_wheel['digests']['sha256']
zipfile.ZipFile(io.BytesIO(_wheel_bytes)).extractall(_demo_tools)
_uv = next(p for p in _demo_tools.rglob('uv') if p.is_file())
_uv.chmod(0o755)
subprocess.run([str(_uv), 'pip', 'install', '--target', str(_demo_packages), 'hyperhtml==0.1.2'], check=True)
sys.path.insert(0, str(_demo_packages))
from hyperhtml import _native

_demo_extension = _demo_root / 'hyper_demo.py'
_demo_extension.write_text('"""Load with `%load_ext hyperhtml.ipython`; run `%%hyper Button`."""\n\nfrom dataclasses import dataclass\nfrom html import escape\nfrom inspect import isawaitable, signature\nfrom keyword import iskeyword\nfrom uuid import uuid4\n\nfrom hyperhtml import _native\n\n\n@dataclass\nclass HyperPreview:\n    html: str\n    python: str\n\n    def _repr_html_(self):\n        ident = f\'hyper-{uuid4().hex}\'\n        return f\'\'\'<div id="{ident}">\n<style>\n#{ident} .panel {{display:none}}\n#{ident} input:checked + label + .panel {{display:block}}\n#{ident} {{display:grid;grid-template-columns:auto auto 1fr;gap:8px}}\n#{ident} input {{position:absolute;opacity:0;width:0}}\n#{ident} label {{grid-row:1;cursor:pointer;padding:6px 12px;border:1px solid #888;border-radius:4px}}\n#{ident} input:checked + label {{background:#ddd;color:#111}}\n#{ident} input:focus-visible + label {{outline:2px solid #268bd2}}\n#{ident} .panel {{grid-row:2;grid-column:1 / -1}}\n#{ident} pre {{overflow:auto;max-height:600px;white-space:pre}}\n</style>\n<input type="radio" name="{ident}" id="{ident}-preview" checked>\n<label for="{ident}-preview">Preview</label>\n<div class="panel"><iframe title="Hyper preview" sandbox="" style="width:100%;height:360px;border:0;background:white" srcdoc="{escape(self.html, quote=True)}"></iframe></div>\n<input type="radio" name="{ident}" id="{ident}-python">\n<label for="{ident}-python">Python</label>\n<div class="panel"><pre><code>{escape(self.python)}</code></pre></div>\n</div>\'\'\'\n\n\ndef load_ipython_extension(ipython):\n    def hyper(line, cell):\n        """Compile a named component and preview it using notebook variables."""\n        name = line.strip()\n        if not name.isidentifier() or iskeyword(name):\n            raise ValueError(\'Provide a component name, for example: %%hyper Button\')\n\n        filename = f\'{name}.hyper\'\n        python = _native.transpile(cell, filename)\n        namespace = ipython.user_ns.copy()\n        exec(compile(python, filename, \'exec\'), namespace)\n        component = namespace[name]\n\n        props = {\n            prop: ipython.user_ns[prop]\n            for prop in signature(component).parameters\n            if prop in ipython.user_ns\n        }\n        rendered = component(**props)\n        if isawaitable(rendered):\n            rendered.close()\n            raise TypeError(\'The Hyper preview requires a synchronous component\')\n\n        ipython.user_ns[name] = component\n        return HyperPreview(str(rendered), python)\n\n    ipython.register_magic_function(hyper, magic_kind=\'cell\', magic_name=\'hyper\')\n\n\ndef unload_ipython_extension(ipython):\n    ipython.magics_manager.magics[\'cell\'].pop(\'hyper\', None)\n')
sys.path.insert(0, str(_demo_root))
%reload_ext hyper_demo
from IPython.display import Javascript, display
display(Javascript('window.hyperHighlightReady = (async () => {\n  const url = URL.createObjectURL(new Blob(["export function notebookGrammar(grammar) {\\n  return {\\n    ...grammar,\\n    name: \'hyper\',\\n    patterns: [\\n      {match: \'^%%hyper\\\\\\\\b.*$\', name: \'keyword.control.hyper\'},\\n      ...grammar.patterns,\\n    ],\\n  };\\n}\\n\\nclass HyperState {\\n  constructor(stack = null) { this.stack = stack; }\\n  clone() { return this; }\\n  equals(other) {\\n    return other instanceof HyperState &&\\n      (this.stack === other.stack || Boolean(this.stack?.equals(other.stack)));\\n  }\\n}\\n\\nexport function tokensProvider(highlighter) {\\n  const grammar = highlighter.getLanguage(\'hyper\');\\n\\n  return {\\n    getInitialState: () => new HyperState(),\\n    tokenize(line, state) {\\n      const result = grammar.tokenizeLine(line, state.stack);\\n      return {\\n        endState: new HyperState(result.ruleStack),\\n        tokens: result.tokens.map(token => ({\\n          startIndex: token.startIndex,\\n          scopes: token.scopes.at(-1),\\n        })),\\n      };\\n    },\\n  };\\n}\\n\\nexport function highlightHistory(root, highlighter) {\\n  const document = root.ownerDocument;\\n  const rendered = new WeakMap();\\n  const observer = new document.defaultView.MutationObserver(refresh);\\n\\n  function refresh() {\\n    observer.disconnect();\\n\\n    try {\\n      for (const code of root.querySelectorAll(\'.card-in pre code.language-hyper\')) {\\n        const source = code.textContent;\\n        if (rendered.get(code) === source) continue;\\n\\n        const template = document.createElement(\'template\');\\n        template.innerHTML = highlighter.codeToHtml(source, {\\n          lang: \'hyper\',\\n          themes: {light: \'github-light\', dark: \'github-dark\'},\\n          defaultColor: false,\\n        });\\n        code.replaceChildren(...template.content.querySelector(\'code\').childNodes);\\n        code.classList.add(\'hyper-history\');\\n        rendered.set(code, source);\\n      }\\n    } finally {\\n      observer.observe(root, {childList: true, characterData: true, subtree: true});\\n    }\\n  }\\n\\n  refresh();\\n  return observer;\\n}\\n\\nexport async function installHyperHighlighting(grammar) {\\n  const {createHighlighter} = await import(\'https://esm.sh/shiki@3.22.0\');\\n  const highlighter = await createHighlighter({\\n    langs: [\'python\', notebookGrammar(grammar)],\\n    themes: await Promise.all([\'/vendor/gh-light-1.json\', \'/vendor/gh-dark-1.json\'].map(async url => {\\n      const response = await fetch(url);\\n      if (!response.ok) throw new Error(`Cannot load SolveIT theme: ${url}`);\\n      return response.json();\\n    })),\\n  });\\n\\n  window.hyperHighlight?.dispose();\\n  window.hyperModelHooks?.forEach(hook => hook.dispose());\\n  document.getElementById(\'hyper-token-colors\')?.remove();\\n\\n  if (!monaco.languages.getLanguages().some(language => language.id === \'hyper\')) {\\n    monaco.languages.register({id: \'hyper\'});\\n  }\\n  const provider = monaco.languages.setTokensProvider(\'hyper\', tokensProvider(highlighter));\\n  const style = document.createElement(\'style\');\\n  style.textContent = `\\n    code.hyper-history span {color:var(--shiki-light);font-style:var(--shiki-light-font-style);font-weight:var(--shiki-light-font-weight)}\\n    .dark code.hyper-history span {color:var(--shiki-dark);font-style:var(--shiki-dark-font-style);font-weight:var(--shiki-dark-font-weight)}\\n  `;\\n  document.head.append(style);\\n  const history = highlightHistory(document.getElementById(\'dialog-container\'), highlighter);\\n\\n  window.hyperHighlight = {\\n    dispose() {\\n      history.disconnect();\\n      style.remove();\\n      provider.dispose();\\n      highlighter.dispose();\\n    },\\n  };\\n\\n  // SolveIT caches magic-to-language mappings when its editor first opens.\\n  if (typeof _langMap !== \'undefined\') _langMap = undefined;\\n  for (const model of monaco.editor.getModels()) {\\n    if (model.getValue().startsWith(\'%%hyper\')) monaco.editor.setModelLanguage(model, \'hyper\');\\n  }\\n}\\n"], {type: \'text/javascript\'}));\n  try {\n    const {installHyperHighlighting} = await import(url);\n    await installHyperHighlighting({"$schema": "https://raw.githubusercontent.com/martinring/tmlanguage/master/tmlanguage.json", "name": "Hyper", "scopeName": "source.hyper", "fileTypes": ["hyper"], "patterns": [{"include": "#comment"}, {"include": "#props"}, {"include": "#keywords"}, {"include": "#html"}], "repository": {"comment": {"name": "comment.block.html", "begin": "<!--", "end": "-->"}, "props": {"patterns": [{"begin": "^(?=[a-z_][a-z_0-9]*:)", "end": "$", "patterns": [{"include": "source.python"}]}, {"match": "^---\\\\s*$", "name": "keyword.control.hyper"}]}, "keywords": {"patterns": [{"comment": "Control flow keywords: must be at line start and line must end with \':\'", "match": "^\\\\s*(if|elif|for|while|with|try|match)\\\\b(.+)(:)\\\\s*$", "captures": {"1": {"name": "keyword.control.flow.python"}, "2": {"patterns": [{"include": "source.python"}]}, "3": {"name": "punctuation.section.block.python"}}}, {"comment": "Bare block continuations: keyword + colon only", "match": "^\\\\s*(else|finally)\\\\s*(:)\\\\s*$", "captures": {"1": {"name": "keyword.control.flow.python"}, "2": {"name": "punctuation.section.block.python"}}}, {"comment": "except/case: keyword at line start, line ends with \':\'", "match": "^\\\\s*(except|case)\\\\b(.+)(:)\\\\s*$", "captures": {"1": {"name": "keyword.control.flow.python"}, "3": {"name": "punctuation.section.block.python"}}}, {"match": "^\\\\s*(end)\\\\s*$", "captures": {"1": {"name": "keyword.control.flow.python"}}}]}, "html": {"patterns": [{"include": "#component-tag"}, {"include": "#html-tag"}, {"include": "#expression"}]}, "component-tag": {"patterns": [{"comment": "Opening named slot: <{...name}>", "begin": "(<)(\\\\{)(\\\\.\\\\.\\\\.)(\\\\w+)(\\\\})", "beginCaptures": {"1": {"name": "punctuation.definition.tag.begin.html"}, "2": {"name": "punctuation.section.interpolation.begin"}, "3": {"name": "keyword.operator.spread.python"}, "4": {"name": "entity.name.tag.slot.hyper"}, "5": {"name": "punctuation.section.interpolation.end"}}, "end": "(>)", "endCaptures": {"1": {"name": "punctuation.definition.tag.end.html"}}}, {"comment": "Closing named slot: </{...name}>", "match": "(</)(\\\\{)(\\\\.\\\\.\\\\.)(\\\\w+)(\\\\})(>)", "captures": {"1": {"name": "punctuation.definition.tag.begin.html"}, "2": {"name": "punctuation.section.interpolation.begin"}, "3": {"name": "keyword.operator.spread.python"}, "4": {"name": "entity.name.tag.slot.hyper"}, "5": {"name": "punctuation.section.interpolation.end"}, "6": {"name": "punctuation.definition.tag.end.html"}}}, {"comment": "Opening component: <{Name}> or <{Name} attrs...>", "begin": "(<)(\\\\{)([A-Z][a-zA-Z0-9]*)(\\\\})", "beginCaptures": {"1": {"name": "punctuation.definition.tag.begin.html"}, "2": {"name": "punctuation.section.interpolation.begin"}, "3": {"name": "support.class.component.html"}, "4": {"name": "punctuation.section.interpolation.end"}}, "end": "(/?>)", "endCaptures": {"1": {"name": "punctuation.definition.tag.end.html"}}, "patterns": [{"include": "#tag-attributes"}]}, {"comment": "Closing component: </{Name}>", "match": "(</)(\\\\{)([A-Z][a-zA-Z0-9]*)(\\\\})(>)", "captures": {"1": {"name": "punctuation.definition.tag.begin.html"}, "2": {"name": "punctuation.section.interpolation.begin"}, "3": {"name": "support.class.component.html"}, "4": {"name": "punctuation.section.interpolation.end"}, "5": {"name": "punctuation.definition.tag.end.html"}}}, {"comment": "Opening plain component: <Component>", "begin": "(<)([A-Z][a-zA-Z0-9]*)", "beginCaptures": {"1": {"name": "punctuation.definition.tag.begin.html"}, "2": {"name": "support.class.component.html"}}, "end": "(/?>)", "endCaptures": {"1": {"name": "punctuation.definition.tag.end.html"}}, "patterns": [{"include": "#tag-attributes"}]}, {"comment": "Closing plain component: </Component>", "match": "(</)([A-Z][a-zA-Z0-9]*)(>)", "captures": {"1": {"name": "punctuation.definition.tag.begin.html"}, "2": {"name": "support.class.component.html"}, "3": {"name": "punctuation.definition.tag.end.html"}}}]}, "html-tag": {"patterns": [{"begin": "(<)([a-z][a-z0-9-]*)", "beginCaptures": {"1": {"name": "punctuation.definition.tag.begin.html"}, "2": {"name": "entity.name.tag.html"}}, "end": "(/?>)", "endCaptures": {"1": {"name": "punctuation.definition.tag.end.html"}}, "patterns": [{"include": "#tag-attributes"}]}, {"match": "(</)([a-z][a-z0-9-]*)(>)", "captures": {"1": {"name": "punctuation.definition.tag.begin.html"}, "2": {"name": "entity.name.tag.html"}, "3": {"name": "punctuation.definition.tag.end.html"}}}]}, "tag-attributes": {"patterns": [{"begin": "([a-zA-Z_][a-zA-Z0-9_-]*)(=)(\\\\{)", "beginCaptures": {"1": {"name": "entity.other.attribute-name.html"}, "2": {"name": "punctuation.separator.key-value.html"}, "3": {"name": "punctuation.section.interpolation.begin"}}, "end": "(\\\\})", "endCaptures": {"1": {"name": "punctuation.section.interpolation.end"}}, "patterns": [{"include": "#python-expr"}]}, {"begin": "([a-zA-Z_][a-zA-Z0-9_-]*)(=)(\\")", "beginCaptures": {"1": {"name": "entity.other.attribute-name.html"}, "2": {"name": "punctuation.separator.key-value.html"}, "3": {"name": "punctuation.definition.string.begin.html"}}, "end": "(\\")", "endCaptures": {"1": {"name": "punctuation.definition.string.end.html"}}, "contentName": "string.quoted.double.html"}]}, "expression": {"begin": "(\\\\{)", "beginCaptures": {"1": {"name": "punctuation.section.interpolation.begin"}}, "end": "(\\\\})", "endCaptures": {"1": {"name": "punctuation.section.interpolation.end"}}, "patterns": [{"include": "#python-expr"}]}, "python-expr": {"patterns": [{"match": "\\\\.\\\\.\\\\.", "name": "keyword.operator.spread.python"}, {"match": "\\\\b(True|False|None)\\\\b", "name": "constant.language.python"}, {"match": "[0-9]+", "name": "constant.numeric.python"}, {"match": "\\"[^\\"]*\\"", "name": "string.quoted.double.python"}, {"match": "\'[^\']*\'", "name": "string.quoted.single.python"}, {"match": "[a-z_][a-z_0-9]*", "name": "variable.other.python"}]}}});\n  } finally { URL.revokeObjectURL(url); }\n})();'))
print('Hyper ready. Run the next cell, then edit and rerun the Hyper cell.')


In [ ]:
names = ['Ada', 'Lin', 'Chris']

In [ ]:
%%hyper Greeting
names: list[str]
---
<main style="font: 20px system-ui; padding: 24px">
<h1>Hyper in SolveIT</h1>
<ul>
    for name in names:
        <li>Hello, {name}!</li>
    end
</ul>
</main>